In [ ]:
#sample code
import gurobipy
from gurobipy import Model
model = Model("test")
print("Gurobi is working!", "\U0001F600")


import cobra
import numpy as np
import scipy.stats as sts
import umap
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from tqdm import tqdm

MAX_SEC = 1000


def create_env_ball(n, metabolites, water_index=0, oxygen_index=1, no_oxygen=True, factor=10, randomSeed = 666):
    np.random.seed(randomSeed)
    samples = np.random.dirichlet(alpha=np.ones(len(metabolites)), size=n)*factor
    env_ball = {}
    for i,v in enumerate(samples):
        env_ball[i] = {metabolites[k]: samples[i][k] for k in range(len(metabolites)) }
    if no_oxygen:
        samples[:, oxygen_index] = 0
    return samples, env_ball


def apply_env(model: cobra.Model,
              env: dict[str, float],
              upper_bound: float = MAX_SEC):
    for rid, conc in env.items():
        if model.reactions.has_id(rid):
            rxn = model.reactions.get_by_id(rid)
            rxn.lower_bound = -abs(conc)
            rxn.upper_bound = upper_bound
    #assure all models have the same access to water, instead of dividing in the distribution
    model.reactions.get_by_id('EX_cpd00001_e0').lower_bound=-10
    solution = model.optimize()
    
    return solution.objective_value


def plot_env_ball(mat, water_index, oxygen_index, color='green', edge_c='k', size=20, line_w=0.25):
    '''
    Plot the 3D UMAP projection of the environment matrix,
    excluding water and oxygen dimensions.
    '''
    plt.style.use('ggplot')
    i = np.arange(mat.shape[1])
    mat = mat[:, (i != water_index) & (i != oxygen_index)]

    # 3D UMAP
    trans = umap.UMAP(n_neighbors=10, random_state=666, n_components=3, min_dist=0.8, metric='cosine').fit(mat)

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(
        trans.embedding_[:, 0],
        trans.embedding_[:, 1],
        trans.embedding_[:, 2],
        s=size,
        alpha=0.6,
        c=color,
        edgecolors=edge_c,
        linewidths=line_w
    )

    ax.set_xlabel('UMAP_1')
    ax.set_ylabel('UMAP_2')
    ax.set_zlabel('UMAP_3')
    plt.tight_layout()
    plt.show()




model = cobra.io.read_sbml_model('C:/Users/drgarza/Documents/GitHub/dFBA/files/models/bifido_example.xml.xml')
# Example use with your model
metabolites = [i.id for i in model.reactions if 'EX_' in i.id]
water_index = metabolites.index('EX_cpd00001_e0')
oxygen_index = metabolites.index('EX_cpd00007_e0')

env_ball, env_ball_dict = create_env_ball(1000, metabolites, water_index, oxygen_index, no_oxygen=True, factor=10)
#plot_env_ball(env_ball, water_index, oxygen_index)


def applyEnvBall(model, env_ball_dict):
    return np.array([apply_env(model, env_ball_dict[i]) for i in range(len(env_ball_dict))])




In [ ]:
import os
os.environ['GRB_LICENSE_FILE'] = '../../content/licenses/gurobi.lic'

In [ ]:
import cobra
import numpy as np
import scipy.stats as sts
import umap
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from tqdm import tqdm

MAX_SEC = 1000

def create_env_ball(n, metabolites, water_index=0, oxygen_index=1, no_oxygen=True, factor=10, randomSeed=666, start_index=0):
    """
    Create an environment ball with a specified number of samples.
    """
    np.random.seed(randomSeed)
    total = n + start_index
    samples_all = np.random.dirichlet(alpha=np.ones(len(metabolites)), size=total) * factor
    samples = samples_all[start_index:]  

    if no_oxygen:
        samples[:, oxygen_index] = 0

    env_ball = {}
    for i, v in enumerate(samples):
        env_ball[start_index + i] = {metabolites[k]: v[k] for k in range(len(metabolites))}

    return samples, env_ball

def apply_env(model: cobra.Model,
              env: dict[str, float],
              upper_bound: float = MAX_SEC):
    for rid, conc in env.items():
        if model.reactions.has_id(rid):
            rxn = model.reactions.get_by_id(rid)
            rxn.lower_bound = -abs(conc)
            rxn.upper_bound = upper_bound
    #assure all models have the same access to water, instead of dividing in the distribution
    model.reactions.get_by_id('EX_cpd00001_e0').lower_bound=-10
    solution = model.optimize()
    fluxes = solution.fluxes
    return solution.objective_value, fluxes




In [ ]:
import os
import tempfile
import pickle
import pandas as pd
import numpy as np
import libsbml
from cobra.io import read_sbml_model

MAX_SEC = 1000

def safe_load_model(filepath):
    doc = libsbml.readSBML(filepath)
    if doc.getNumErrors() > 0:
        print(f"SBML 错误: {filepath}")
        for i in range(doc.getNumErrors()):
            err = doc.getError(i)
            print(f" - ({err.getSeverity()}) {err.getMessage()}")
        return None

    sbml_str = libsbml.writeSBMLToString(doc)
    with tempfile.NamedTemporaryFile("w", delete=False, suffix=".xml") as tmp:
        tmp.write(sbml_str)
        tmp_path = tmp.name

    return read_sbml_model(tmp_path)

def read_exchange_reactions(tsv_path):
    with open(tsv_path, 'r') as f:
        lines = f.readlines()
    return [line.strip().split('\t')[0] for line in lines[1:] if line.strip()]


def create_env_ball(n, metabolites, water_index=0, oxygen_index=1, no_oxygen=True, factor=10, randomSeed=666, start_index=0):
    np.random.seed(randomSeed)
    total = n + start_index
    samples_all = np.random.dirichlet(alpha=np.ones(len(metabolites)), size=total) * factor
    samples = samples_all[start_index:]  

    if no_oxygen:
        samples[:, oxygen_index] = 0

    env_ball = {}
    for i, v in enumerate(samples):
        env_ball[start_index + i] = {metabolites[k]: v[k] for k in range(len(metabolites))}

    return samples, env_ball

def apply_env(env, model):
    for reaction_id in EXCHANGE_REACTIONS:
        if model.reactions.has_id(reaction_id):
            model.reactions.get_by_id(reaction_id).lower_bound = -1 * env[reaction_id]
    solution = model.optimize()
    fluxes = [solution.fluxes[reaction_id] if model.reactions.has_id(reaction_id) else 0.0 for reaction_id in EXCHANGE_REACTIONS]
    return solution.objective_value, fluxes

def simulate_all_envs(env_ball, models, EXCHANGE_REACTIONS):
    result_dict = {}
    for env_idx, env in env_ball.items():
        result_dict[env_idx] = {}
        for model_id, model in models.items():
            print(f"\n Simulating model {model_id} in environment {env_idx}")
            model_copy = model.copy()
            biomass, fluxes = apply_env(env, model_copy)
            result_dict[env_idx][model_id] = (biomass, fluxes)
            print(f" Biomass = {biomass}")
            nonzero_fluxes = [(rid, flux) for rid, flux in zip(EXCHANGE_REACTIONS, fluxes) if abs(flux) > 1e-6]
            if nonzero_fluxes:
                print(f" Nonzero exchange fluxes: {nonzero_fluxes[:5]}")
            else:
                print(" All exchange fluxes are 0")
    return result_dict

def extract_target_flux_df(sim_result, target_reacts, EXCHANGE_REACTIONS):
    rows = []
    for env_idx, model_results in sim_result.items():
        for model_id, (biomass, fluxes_all) in model_results.items():
            row = {"env_idx": env_idx, "model_id": model_id, "biomass": biomass}
            for name, rxn_id in target_reacts.items():
                if rxn_id in EXCHANGE_REACTIONS:
                    idx = EXCHANGE_REACTIONS.index(rxn_id)
                    row[name] = fluxes_all[idx]
                else:
                    row[name] = None
            rows.append(row)
    return pd.DataFrame(rows)


if __name__ == "__main__":
    models_dir = "d:/Electromics-project/models/fixed_model"
    ex_react_path = "../../models/group_medium_exchanges.tsv"

    # 1. 加载 exchange reaction ID 列表
    EXCHANGE_REACTIONS = read_exchange_reactions(ex_react_path)
    print(f" Loaded {len(EXCHANGE_REACTIONS)} exchange reactions")

    # 2. 生成随机环境（1000 个）
    samples, env_ball = create_env_ball(
        n=1000,
        metabolites=EXCHANGE_REACTIONS,
        water_index=0,
        oxygen_index=1,
        no_oxygen=True,
        factor=10,
        randomSeed=666
    )
    print(f" Generated {len(env_ball)} environments")

    # 3. 加载模型
    model_files = sorted([f for f in os.listdir(models_dir) if f.endswith(".xml")])
    models = {}
    for filename in model_files:
        path = os.path.join(models_dir, filename)
        model_id = os.path.splitext(filename)[0]
        print(f" 尝试加载: {filename}")
        model = safe_load_model(path)
        if model:
            models[model_id] = model
            print(f" 成功加载模型: {model_id}")
        else:
            print(f" 加载失败: {model_id}")

    # 4. 模拟所有环境
    result = simulate_all_envs(env_ball, models, EXCHANGE_REACTIONS)

    # 5. 可选保存
    with open("simulation_result_new.pkl", "wb") as f:
        pickle.dump(result, f)
    print("结果已保存")
